# Selección y construcción de variables predictoras

Se parte desde `data/dataset_ml_con_respuesta.parquet` y define el conjunto final de variables predictoras que se usarán para
entrenar los modelos de clasificación de la siguiente etapa del laboratorio.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_parquet("./data/dataset_ml_con_respuesta.parquet")
df.head()

## 1. Variables excluidas

`NDCI`, `B04` y `B05` se excluyen del conjunto de predictores por fuga de información. Se mantienen en
el DataFrame solo como referencia/trazabilidad.

## 2. Conjunto de variables predictoras

### 3.1 – 3.2 Definición y justificación de cada variable

| Variable | Tipo | Qué representa | Por qué ayuda a detectar cianobacteria |
|---|---|---|---|
| `B03` (Verde) | Banda espectral | Reflectancia en el verde (~560 nm). | Sensible a la absorción de pigmentos fotosintéticos; también es la base del NDWI (usado además como máscara de agua), por lo que aporta información sobre la condición del agua. No interviene en la fórmula del NDCI, así que es segura de usar. |
| `B08` (Infrarrojo cercano) | Banda espectral | Reflectancia en el NIR (~842 nm). | El agua limpia absorbe casi todo el NIR; aumentos de reflectancia en NIR sobre agua suelen indicar presencia de materia en suspensión, incluida biomasa algal densa. No interviene en la fórmula del NDCI. |
| `NDVI` | Índice espectral (derivado de B04, B08) | Biomasa fotosintética. | Sobre agua, capta la presencia de organismos fotosintéticos en superficie de forma independiente al NDCI, ya que combina B04 con B08 (no con B05). En la Parte I se encontró una correlación fuerte pero no perfecta entre NDVI y NDCI (r = 0.93 en Atitlán, r = 0.89 en Amatitlán a nivel de imagen), lo que sugiere señal complementaria, aunque en Atitlán esta relación debe interpretarse con cautela por la inestabilidad numérica del NDCI en aguas muy claras. |
| `NDWI` | Índice espectral (derivado de B03, B08) | Contenido/pureza del agua; también se usó como criterio de máscara de agua (NDWI > 0). | En Amatitlán, el notebook 03 encontró una correlación negativa clara entre NDWI y NDCI (r = −0.72 a nivel de imagen): agua más turbia se asocia con más cianobacteria. En Atitlán la relación no fue significativa, pero se mantiene como variable de contexto. |
| `lago` (codificada) | Categórica → dummy/one-hot | Identificador del cuerpo de agua. | Los dos lagos tienen regímenes limnológicos muy distintos (profundidad, tiempo de renovación, uso de suelo en la cuenca, transparencia del agua), documentado extensamente en la Parte I. Incluirla permite que el modelo aprenda relaciones espectrales condicionales al lago — esto es especialmente relevante dado que en Atitlán el NDCI es numéricamente más inestable. |
| `mes` (de la fecha) | Categórica cíclica (o numérica 1-12) | Estacionalidad del muestreo. | La Parte I mostró indicios de mayor floración en transiciones estacionales (ene-feb, jun-jul), particularmente en Amatitlán durante inicio de lluvias. Aporta contexto temporal sin usar la fecha exacta (que sería casi un identificador único por imagen). |
| `lon`, `lat` | Coordenadas espaciales | Posición geográfica del píxel dentro del lago. | El notebook 03 identificó zonas persistentes de acumulación (p. ej. cuenca sureste de Amatitlán con el mayor incremento de NDCI). Incluir coordenadas permite al modelo aprender patrones espaciales de riesgo, aunque se debe tener cuidado de no sobreajustar a ubicaciones exactas vistas en entrenamiento (ver nota de validación espacial al final). |


## 3. Ingeniería de características adicionales

### 3.3 Variables nuevas propuestas y su justificación


In [ ]:
# 1. Distancia al centro del lago (proxy de "aguas abiertas" vs. "cercano a orilla/afluentes")
centros = df.groupby("lago")[["lon", "lat"]].transform("mean")
df["dist_centro_lago"] = np.sqrt((df["lon"] - centros["lon"])**2 + (df["lat"] - centros["lat"])**2)

# 2. Mes del año (estacionalidad) y variables cíclicas (para no romper la continuidad dic->ene)
df["mes"] = df["fecha"].dt.month
df["mes_sin"] = np.sin(2 * np.pi * df["mes"] / 12)
df["mes_cos"] = np.cos(2 * np.pi * df["mes"] / 12)

# 3. Época (seca / lluviosa) - basada en el patrón climático de Guatemala
#    Época seca: noviembre-abril | Época lluviosa: mayo-octubre
df["epoca_lluviosa"] = df["mes"].isin([5, 6, 7, 8, 9, 10]).astype(int)

# 4. Brillo general (proxy simple de turbidez/nubosidad residual), usando solo bandas permitidas
df["brillo_B03_B08"] = (df["B03"] + df["B08"]) / 2

print("Nuevas variables creadas: dist_centro_lago, mes, mes_sin, mes_cos, epoca_lluviosa, brillo_B03_B08")
df[["dist_centro_lago", "mes", "mes_sin", "mes_cos", "epoca_lluviosa", "brillo_B03_B08"]].describe()


**Justificación de cada variable nueva:**

- **`dist_centro_lago`**: aproxima si un píxel está en aguas abiertas o cerca de la orilla/
  afluentes, sin depender de un polígono externo. El notebook 03 mostró que la floración en
  Amatitlán se concentra de forma diferenciada entre la cuenca ancha (sureste) y el brazo
  angosto (noroccidental) — una variable de posición relativa dentro del lago puede ayudar
  a capturar ese contraste de forma genérica, aplicable a ambos lagos.
- **`mes`, `mes_sin`, `mes_cos`**: la codificación cíclica (seno/coseno) evita el problema de
  tratar diciembre (mes 12) y enero (mes 1) como extremos opuestos en una escala lineal,
  cuando en realidad son consecutivos en el calendario. Relevante porque los picos
  observados en la Parte I ocurren cerca de esas transiciones diciembre-enero y junio-julio.
- **`epoca_lluviosa`**: variable binaria simple y directamente interpretable, alineada con
  la hipótesis discutida en la Parte I sobre el efecto de "primer lavado" (*first flush*) de
  nutrientes al iniciar las lluvias.
- **`brillo_B03_B08`**: combinación simple de dos bandas ya permitidas (no usa B04 ni B05),
  como proxy adicional de condiciones atípicas de reflectancia (nubosidad residual no
  capturada por el filtro de nubes del notebook 1, oleaje, sedimentos en suspensión).

Todas las variables nuevas se construyen exclusivamente a partir de bandas/coordenadas/fecha
ya disponibles y **no dependen de B04, B05 ni NDCI**, por lo que no reintroducen fuga de
información.


## 4. Conjunto final de predictores


In [ ]:
PREDICTORAS_NUMERICAS = [
    "B03", "B08", "NDVI", "NDWI",
    "lon", "lat", "dist_centro_lago",
    "mes_sin", "mes_cos", "epoca_lluviosa",
    "brillo_B03_B08",
]
PREDICTORAS_CATEGORICAS = ["lago"]

X = pd.get_dummies(df[PREDICTORAS_NUMERICAS + PREDICTORAS_CATEGORICAS],
                    columns=PREDICTORAS_CATEGORICAS, drop_first=True)
y = df["cianobacteria_alta"]

print(f"Matriz de predictores final: {X.shape[0]:,} filas x {X.shape[1]} columnas")
print(f"Columnas: {list(X.columns)}")

X.to_parquet("./data/X_predictoras.parquet", index=False)
y.to_frame().to_parquet("./data/y_respuesta.parquet", index=False)
print("\nGuardado: ./data/X_predictoras.parquet y ./data/y_respuesta.parquet")
